In [2]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad

In [3]:
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=80)  # low dpi (dots per inch) yields small inline figures
sc.logging.print_version_and_date()
sc.set_figure_params(figsize=(12,12))

In [4]:
adata = sc.read_h5ad(snakemake.input[0])
adata.X = adata.X.astype('f')
adata

In [5]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=5)

In [6]:
data = adata
data

In [7]:
data.var['mt'] = data.var["gene_symbols"].str.lower().str.startswith('mt').astype(bool)  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(data, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [8]:
sc.pl.violin(data, ['n_genes', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [9]:
sc.pl.scatter(data, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(data, x='total_counts', y='n_genes')

In [10]:
if 'droplet_type' in adata.obs.columns:
    data = data[data.obs['droplet_type'] == 'singlet',:]
data = data[data.obs['pct_counts_mt'] < 15,:]
data = data[data.obs['total_counts'] > 1e3,:]
data = data[data.obs['total_counts'] < 25e3,:]
sc.pl.scatter(data, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(data, x='total_counts', y='n_genes')

In [11]:
sc.pl.violin(data, ['n_genes', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, save="_post_filter.pdf")


In [12]:
data

In [13]:
data.X 

In [15]:
sc.pp.normalize_per_cell(data, counts_per_cell_after=1.0e4)
sc.pl.highest_expr_genes(data, n_top=10, gene_symbols='gene_symbols')

In [16]:
sc.pp.log1p(data)

In [17]:
sc.pp.highly_variable_genes(data, n_top_genes=2000)

In [18]:
sc.pl.highly_variable_genes(data)

In [19]:
sc.tl.pca(data)

In [20]:
sc.pl.pca_scatter(data, color=['total_counts', 'n_genes','pct_counts_mt'])

In [21]:
sc.pp.neighbors(data)
sc.tl.leiden(data)
sc.tl.umap(data)

In [22]:
sc.pl.umap(data, color=['total_counts', 'n_genes','pct_counts_mt', 'sample_id'], s=50)

In [41]:
if 'Sample_Group' in adata.obs.columns:
    sc.pl.umap(data, color=['sample_id', 'Sample_Group'], s=50)

In [40]:
sublib = data.obs_names.str.extract(r'__s(\d+)$', expand=False)
if sublib.isna().any():
    n_missing = int(sublib.isna().sum())
    raise ValueError(f"Could not extract Parse sublibrary from {n_missing} barcodes")
data.obs['sublib'] = pd.Categorical(sublib)
sc.pl.umap(data, color=['sublib'], s=50, alpha=0.4)


In [23]:
sc.pl.umap(data, color=['leiden'], s=50)

In [24]:
#import scvelo as scv

In [25]:
#scv.pp.moments(data, n_pcs=30, n_neighbors=30)

In [26]:
#scv.tl.velocity(data)

In [27]:
#scv.tl.velocity_graph(data)

In [28]:
#scv.pl.velocity_embedding_stream(data, basis='umap')

In [42]:
data
with ad.settings.override(allow_write_nullable_strings=True):
    data.write(snakemake.output['preprocessed'])
